# ST-OMR Meter V4-1 Learned Numerator Specialist

Fixed 3-fold family-disjoint OOF training on the exact accepted V4-0 27-crop artifact. No Teacher Gold adaptation-validation, D10, TEST, runtime, Resolver or production access.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from hashlib import sha256
import json, shutil, subprocess, sys

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_REF = 'fix/meter-v4-1-learned-numerator-specialist'
WORK_ROOT = Path('/content/st-omr-meter-v4-1')
REPO_DIR = WORK_ROOT / 'repo'
PARENT_V4_0_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/meter-v4-0-numerator-representation-audit-8641fc45ae0e')
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git','clone','--branch',REPO_REF,'--single-branch','--filter=blob:none',REPO_URL,str(REPO_DIR)], check=True)
git_commit_sha = subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if len(git_commit_sha) != 40 or any(ch not in '0123456789abcdef' for ch in git_commit_sha):
    raise RuntimeError('Git commit SHA is not canonical lowercase SHA-1')
repository_binding = sha256(('git-commit-sha1:' + git_commit_sha).encode('ascii')).hexdigest()
OUTPUT_ROOT = DRIVE_RUNS_ROOT / f'meter-v4-1-learned-numerator-specialist-{repository_binding[:12]}'

required = [PARENT_V4_0_ROOT/'result.json', PARENT_V4_0_ROOT/'COMPLETE', PARENT_V4_0_ROOT/'crops']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing accepted V4-0 parent artifact(s): ' + repr(missing))

print(json.dumps({
    'experiment': 'meter-v4-1-learned-numerator-specialist-v1',
    'git_commit_sha': git_commit_sha,
    'repository_sha256_binding': repository_binding,
    'parent_v4_0_root': str(PARENT_V4_0_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'planned_parent_crops': 27,
    'folds': 3,
    'epochs_per_training': 160,
    'deterministic_repeat_per_fold': 2,
    'd10_opened': False,
    'teacher_adaptation_validation_evaluated': False,
    'test_opened': False,
}, indent=2))


## Install exact pinned training runtime

This installs the repository-pinned CPU training dependencies. The experiment itself runs in a fresh subprocess, so an older Torch already loaded by the notebook kernel cannot contaminate the V4-1 run.


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
    '-r', str(REPO_DIR/'requirements-training.txt')
], check=True)
runtime = subprocess.run([
    sys.executable, '-c', 'import torch; print(torch.__version__)'
], check=True, capture_output=True, text=True).stdout.strip()
print('SUBPROCESS TORCH:', runtime)
if runtime != '2.13.0+cpu':
    raise RuntimeError(f'Pinned Torch mismatch after install: {runtime}')


## Run / live progress


In [ ]:
result_path = OUTPUT_ROOT / 'result.json'
complete_path = OUTPUT_ROOT / 'COMPLETE'
if result_path.is_file() and complete_path.is_file():
    print('V4-1 already COMPLETE; reusing immutable result.')
else:
    if OUTPUT_ROOT.exists():
        raise RuntimeError(f'Incomplete V4-1 output exists; do not overwrite: {OUTPUT_ROOT}')
    part = OUTPUT_ROOT.with_name('.' + OUTPUT_ROOT.name + '.part')
    if part.exists():
        raise RuntimeError(f'Incomplete V4-1 temporary output exists; do not overwrite: {part}')
    command = [
        sys.executable, '-u', str(REPO_DIR/'tools/meter_v4_1_numerator_specialist_runner.py'),
        '--repository-root', str(REPO_DIR),
        '--parent-v4-0-root', str(PARENT_V4_0_ROOT),
        '--output-root', str(OUTPUT_ROOT),
    ]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'V4-1 runner failed with exit code {code}')


## Bounded result


In [ ]:
if not result_path.is_file() or not complete_path.is_file():
    raise RuntimeError('V4-1 result/COMPLETE missing')
result = json.loads(result_path.read_text(encoding='ascii'))
print('==============================================')
print('V4-1 OOF SUMMARY')
print('==============================================')
print(json.dumps(result['oof_summary'], indent=2, ensure_ascii=False))
print('\n==============================================')
print('V4-1 DECISION')
print('==============================================')
print(json.dumps(result['decision'], indent=2, ensure_ascii=False))
print('\n==============================================')
print('DETERMINISM + SAFETY')
print('==============================================')
print(json.dumps({
    'determinism_repeat_pass': result['determinism']['repeat_pass'],
    'total_optimizer_steps': result['total_optimizer_steps'],
    'teacher_adaptation_validation_evaluated': result['data_surface']['teacher_adaptation_validation_evaluated'],
    'teacher_adaptation_validation_images_decoded': result['data_surface']['teacher_adaptation_validation_images_decoded'],
    'd10_opened': result['data_surface']['d10_opened'],
    'test_opened': result['data_surface']['test_opened'],
    'runtime_connected': result['runtime_connected'],
    'production_promotion_authorized': result['production_promotion_authorized'],
}, indent=2))
